# The interactive by-drug figures, and the code that writes them

Two standalone HTML pages, same axes and same interaction, differing in **where each drug's DRM
list comes from**:

| file | drug &rarr; mutations |
|---|---|
| `de_range_by_drug_interactive.html` | **Stanford &times; IAS-USA.** Substitutions from `ms0_5/*/data/drug_resistant_pair_*_stanford.tsv` (Stanford HIVDB penalty-score tables, converted to Position/Consensus/DRM in commit `0b74915`), assigned to a drug when the position is on that drug's IAS-USA list. Stanford gives no per-drug split, so the drug assignment is by position. |
| `de_range_by_drug_interactive_IAS.html` | **IAS-USA alone.** Substitutions *and* drug assignment both read off the IAS-USA 2025 figures (*Topics in Antiviral Medicine* 33(2):457&ndash;465, pp. 462&ndash;463), which name the exact mutants per drug rather than just the codons. |

Both pages are built the same way: take the drug's DRM substitutions, form every pair at distinct
positions, map into the reduced alphabet, drop pairs never co-observed in the MSA (bivariate
marginal 0), then plot each pair's &Delta;E double range with a live &Delta;&Delta;E threshold.

The IAS-USA version is the cleaner object for a manuscript &mdash; one source, and the drug
assignment is stated by that source rather than inferred from a position match. The Stanford
version reaches more pairs, because its substitution lists are longer.

## Setup — alignments, J, consensus

In [1]:
import sys
import csv
import json
import itertools
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = next((c for c in [cwd, *cwd.parents] if (c / "utilities" / "functions.py").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")
sys.path.append(str(repo_root))

import utilities.functions as functions

importlib.reload(functions)

data_root = repo_root / "ms0_5"
out_dir = Path.cwd()


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2]."""
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


CFG = {
    'IN': dict(seq='IN/data/in.reduce4.seq', cons='IN/data/in.consensus.reduce4.seq',
               redux='IN/data/in.reduce4.redux', offset=1, J='IN/data/J.npy',
               min_pos=1, max_pos=263),
    'PR': dict(seq='PR/data/pr.exper.reduce4.seq', cons='PR/data/pr.consensus.reduce4.seq',
               redux='PR/data/pr.reduce4.redux', offset=0, J='PR/data/J_PR.npy',
               min_pos=1, max_pos=99),
    'RT': dict(seq='RT/data/rt.reduce4.seq', cons='RT/data/rt.consensus.reduce4.seq',
               redux='RT/data/rt.reduce4.redux', offset=0, J='RT/data/J_RT.npy',
               min_pos=39, max_pos=226),
}
PROTEINS = list(CFG)

_AA_CODE = np.full(256, 4, dtype=np.uint8)
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i

for P, cfg in CFG.items():
    seqs = functions.read_seq(str(data_root / cfg['seq']))
    cfg['redux'] = functions.get_redu_dict(str(data_root / cfg['redux']), cfg['offset'])
    cfg['Jm'] = build_J_matrix(str(data_root / cfg['J']), cfg['min_pos'], cfg['max_pos'])
    cfg['L'] = L = cfg['max_pos'] - cfg['min_pos'] + 1
    N = len(seqs)
    raw = np.frombuffer("".join(seqs).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    cfg['codes'] = _AA_CODE[raw.reshape(N, L)]
    oh = np.zeros((N * L, 5), dtype=np.float32)
    oh[np.arange(N * L), cfg['codes'].ravel()] = 1.0
    cfg['onehot'] = oh.reshape(N, L * 5)
    cons = functions.read_seq(str(data_root / cfg['cons']))[0].strip()
    assert len(cons) == L, f"{P}: consensus length {len(cons)} != {L}"
    cfg['cons_codes'] = _AA_CODE[np.frombuffer(cons.encode(), dtype=np.uint8)]
    print(f"{P}: {N:>6,} sequences   positions {cfg['min_pos']}-{cfg['max_pos']}  (L = {L})")

IN:  1,220 sequences   positions 1-263  (L = 263)
PR:  5,710 sequences   positions 1-99  (L = 99)
RT: 19,194 sequences   positions 39-226  (L = 188)


### ΔE double range and ΔΔE on consensus

`metrics` returns both numbers the pages need. The ΔΔE reduces to the p1–p2 coupling block once the
background terms cancel:

    ddE = [J(wt1,wt2) - J(mt1,mt2)] - [J(wt1,c2) - J(mt1,c2)] - [J(c1,wt2) - J(c1,mt2)]

with c1, c2 the consensus residues at the two positions. That reproduces the ms0_5 tables —
the assertion below checks G140S-Q148H against its published 8.51.

In [2]:
def reduced_mutation(P, sub):
    """'T66I' -> (0-based position, wt index, mt index), or None if not representable.

    None covers three real losses: position outside the modelled range, substitution missing from
    the reduction dictionary, or consensus and mutant landing in the same reduced class.
    """
    cfg = CFG[P]
    try:
        wt, pos, mt = functions.split_pair(functions.unreduced_to_reduced(cfg['redux'], sub))
    except Exception:
        return None
    if '-' in (wt, mt) or wt == mt:
        return None
    if not (cfg['min_pos'] <= pos <= cfg['max_pos']):
        return None
    return (pos - cfg['min_pos'], "ABCD".index(wt), "ABCD".index(mt))


def _site_energies(onehot, Jm, p, excluded):
    A = Jm[p].copy()
    A[list(excluded)] = 0.0
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def metrics(P, a, b):
    """(95-5 percentile range of dE double over the alignment, mean dE double, ddE on consensus)."""
    cfg = CFG[P]
    (p1, wt1, mt1), (p2, wt2, mt2) = a, b
    S1 = _site_energies(cfg['onehot'], cfg['Jm'], p1, (p1, p2))
    S2 = _site_energies(cfg['onehot'], cfg['Jm'], p2, (p1, p2))
    J12 = cfg['Jm'][p1, p2, :, :4].astype(np.float64)
    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    de = M[:, wt1, wt2] - M[:, mt1, mt2]
    lo, hi = np.percentile(de, [5, 95])

    c1, c2 = cfg['cons_codes'][p1], cfg['cons_codes'][p2]
    J = cfg['Jm'][p1, p2].astype(np.float64)
    dde = ((J[wt1, wt2] - J[mt1, mt2])
           - (J[wt1, c2] - J[mt1, c2])
           - (J[c1, wt2] - J[c1, mt2]))
    return float(hi - lo), float(de.mean()), float(dde)


_a, _b = reduced_mutation('IN', 'G140S'), reduced_mutation('IN', 'Q148H')
_rng, _mean, _dde = metrics('IN', _a, _b)
print(f"IN G140S-Q148H: ddE on consensus = {_dde:.3f}  (ms0_5 tables give 8.51), "
      f"dE double range = {_rng:.2f}")
assert abs(_dde - 8.51) < 0.01

IN G140S-Q148H: ddE on consensus = 8.509  (ms0_5 tables give 8.51), dE double range = 10.26


## Source A — Stanford substitutions at IAS-USA positions

`DRUG_POSITIONS` is the IAS-USA 2025 codon list per drug, as used throughout these notebooks.
`STANFORD` is the substitution table already in the repo. A drug's DRM set is the Stanford
substitutions sitting at its IAS-USA positions, with RT's NRTI and NNRTI tables kept apart.

In [3]:
DRUG_POSITIONS = {
    'IN': {'bictegravir': [118, 138, 140, 148, 153, 263],
           'cabotegravir': [66, 74, 97, 118, 138, 140, 148, 153, 155, 263],
           'dolutegravir': [118, 138, 140, 147, 148, 153, 155, 263],
           'elvitegravir': [66, 92, 97, 118, 121, 138, 140, 147, 148, 155, 263],
           'raltegravir': [74, 92, 97, 118, 121, 138, 140, 143, 148, 155, 263]},
    'PR': {'atazanavir': [10, 20, 24, 32, 33, 46, 48, 50, 53, 54, 73, 82, 84, 85, 88, 90],
           'darunavir': [11, 32, 33, 47, 50, 54, 74, 76, 84, 89],
           'fosamprenavir': [10, 32, 46, 47, 50, 54, 73, 76, 82, 84, 90],
           'indinavir': [10, 20, 24, 32, 36, 46, 54, 71, 73, 76, 77, 82, 84, 90],
           'lopinavir': [10, 20, 24, 32, 33, 46, 47, 50, 53, 54, 71, 73, 76, 82, 84, 90],
           'nelfinavir': [10, 30, 36, 46, 71, 77, 82, 84, 88, 90],
           'saquinavir': [10, 24, 48, 54, 62, 71, 73, 77, 82, 84, 90],
           'tipranavir': [10, 33, 36, 43, 46, 47, 54, 58, 69, 74, 82, 83, 84, 89]},
    'RT': {'zidovudine': [41, 67, 70, 210, 215, 219],
           'stavudine': [41, 65, 67, 70, 210, 215, 219],
           'didanosine': [65, 74], 'abacavir': [65, 74, 115, 184],
           'emtricitabine/lamivudine': [65, 184], 'tenofovir': [65, 70],
           'TAMs': [41, 70, 210, 215, 219],
           '69 insertion complex': [41, 62, 69, 70, 210, 215, 219],
           '151 complex': [62, 75, 77, 116, 151],
           'doravirine': [98, 106, 188, 190, 225, 227, 230, 234, 318],
           'efavirenz': [100, 101, 103, 106, 108, 181, 188, 190, 225, 230],
           'etravirine': [90, 98, 100, 101, 106, 138, 179, 181, 190, 230],
           'nevirapine': [100, 101, 103, 106, 108, 181, 188, 190, 230],
           'rilpivirine': [100, 101, 138, 179, 181, 188, 221, 227, 230]},
}
NNRTI_DRUGS = {'doravirine', 'efavirenz', 'etravirine', 'nevirapine', 'rilpivirine'}
COMPLEXES = {'TAMs', '69 insertion complex', '151 complex'}

STANFORD_FILES = {
    'IN': [('IN/data/drug_resistant_pair_INSTI_stanford.tsv', 'INSTI')],
    'PR': [('PR/data/drug_resistant_pair_PI_stanford.tsv', 'PI')],
    'RT': [('RT/data/drug_resistant_pair_stanford_NRTI.tsv', 'NRTI'),
           ('RT/data/drug_resistant_pair_stanford_NNRTI.tsv', 'NNRTI')],
}

STANFORD = {}
for P in PROTEINS:
    rows = []
    for fn, cls in STANFORD_FILES[P]:
        for r in csv.DictReader(open(data_root / fn, newline=''), delimiter='\t'):
            rows.append((int(r['Position']), r['Consensus'].strip(), r['DRM'].strip(), cls))
    STANFORD[P] = rows

SOURCE_A = {P: {} for P in PROTEINS}
for P in PROTEINS:
    for drug, plist in DRUG_POSITIONS[P].items():
        want = ('NNRTI' if drug in NNRTI_DRUGS else 'NRTI') if P == 'RT' else None
        SOURCE_A[P][drug] = [f"{wt}{pos}{mt}" for pos, wt, mt, cls in STANFORD[P]
                             if pos in plist and (want is None or cls == want)]

print({P: sum(len(v) for v in SOURCE_A[P].values()) for P in PROTEINS},
      "drug-substitution entries")

{'IN': 115, 'PR': 254, 'RT': 316} drug-substitution entries


## Source B — IAS-USA 2025 alone

Transcribed from the mutation figures on pages 462–463 of the 2025 update, read off the PDF with
per-glyph x coordinates so each mutant letter attaches to the codon column printed above it. The
IAS-USA figures give the substitutions **per drug**, which is what makes this version
single-source: nothing here is inferred from a position match.

Two things the figures carry that the model cannot:

* the **69 insertion** itself — an insertion, not a substitution, so the 69 insertion complex is
  represented by its accompanying substitutions only;
* **L74I for cabotegravir** is marked subtype A6 only in the figure; it is kept, since the
  alignment is not subtype-partitioned here.

In [4]:
SOURCE_B = {
    'IN': {
        'bictegravir':  ['G118R', 'E138A', 'E138K', 'E138T', 'G140A', 'G140C', 'G140R', 'G140S',
                         'Q148H', 'Q148K', 'Q148R', 'S153F', 'S153Y', 'R263K'],
        'cabotegravir': ['T66K', 'L74I', 'T97A', 'G118R', 'E138A', 'E138K', 'E138T',
                         'G140A', 'G140C', 'G140R', 'G140S', 'Q148H', 'Q148K', 'Q148R',
                         'S153F', 'S153Y', 'N155H', 'R263K'],          # L74I: subtype A6 only
        'dolutegravir': ['G118R', 'E138A', 'E138K', 'E138T', 'G140A', 'G140C', 'G140R', 'G140S',
                         'S147G', 'Q148H', 'Q148K', 'Q148R', 'S153F', 'S153Y', 'N155H', 'R263K'],
        'elvitegravir': ['T66I', 'T66A', 'T66K', 'E92Q', 'E92G', 'T97A', 'G118R', 'F121Y',
                         'E138A', 'E138K', 'G140A', 'G140C', 'G140S', 'S147G',
                         'Q148H', 'Q148K', 'Q148R', 'N155H', 'R263K'],
        'raltegravir':  ['L74M', 'E92Q', 'T97A', 'G118R', 'F121Y', 'E138A', 'E138K',
                         'G140A', 'G140C', 'G140S', 'Y143R', 'Y143H', 'Y143C',
                         'Q148H', 'Q148K', 'Q148R', 'N155H', 'R263K'],
    },
    'PR': {
        'atazanavir':   ['L10F', 'K20T', 'L24I', 'V32I', 'L33F', 'M46I', 'M46L', 'G48V', 'I50L',
                         'F53L', 'F53Y', 'I54L', 'I54V', 'I54M', 'I54T', 'I54A', 'I54S',
                         'G73C', 'G73S', 'G73T', 'G73A', 'V82A', 'V82T', 'V82F', 'V82L', 'V82M',
                         'V82S', 'I84V', 'I85V', 'N88S', 'L90M'],
        'darunavir':    ['V11I', 'V32I', 'L33F', 'I47V', 'I50V', 'I54M', 'I54L', 'T74P', 'L76V',
                         'I84V', 'L89V'],
        'fosamprenavir': ['L10F', 'L10I', 'L10R', 'L10V', 'V32I', 'M46I', 'M46L', 'I47V', 'I50V',
                          'I54L', 'I54V', 'I54M', 'G73S', 'L76V', 'V82A', 'V82F', 'V82S', 'V82T',
                          'I84V', 'L90M'],
        'indinavir':    ['L10I', 'L10R', 'L10V', 'K20M', 'K20R', 'L24I', 'V32I', 'M36I',
                         'M46I', 'M46L', 'I54V', 'A71V', 'A71T', 'G73S', 'G73A', 'L76V',
                         'V77I', 'V82A', 'V82F', 'V82T', 'I84V', 'L90M'],
        'lopinavir':    ['L10F', 'L10I', 'L10R', 'L10V', 'K20M', 'K20R', 'L24I', 'V32I', 'L33F',
                         'M46I', 'M46L', 'I47V', 'I47A', 'I50V', 'F53L', 'I54V', 'I54L', 'I54A',
                         'I54M', 'I54T', 'I54S', 'A71V', 'A71T', 'G73S', 'L76V', 'V82A', 'V82F',
                         'V82T', 'V82S', 'I84V', 'L90M'],
        'nelfinavir':   ['L10F', 'L10I', 'D30N', 'M36I', 'M46I', 'M46L', 'A71V', 'A71T', 'V77I',
                         'V82A', 'V82F', 'V82T', 'V82S', 'I84V', 'N88D', 'N88S', 'L90M'],
        'saquinavir':   ['L10I', 'L10R', 'L10V', 'L24I', 'G48V', 'I54V', 'I54L', 'I62V',
                         'A71V', 'A71T', 'G73S', 'V77I', 'V82A', 'V82F', 'V82T', 'V82S',
                         'I84V', 'L90M'],
        'tipranavir':   ['L10V', 'L33F', 'M36I', 'M36L', 'M36V', 'K43T', 'M46L', 'I47V',
                         'I54A', 'I54M', 'I54V', 'Q58E', 'H69K', 'H69R', 'T74P', 'V82L', 'V82T',
                         'N83D', 'I84V', 'L89I', 'L89M', 'L89V'],
    },
    'RT': {
        'zidovudine':   ['M41L', 'D67N', 'K70R', 'L210W', 'T215Y', 'T215F', 'K219Q', 'K219E'],
        'stavudine':    ['M41L', 'K65R', 'K65E', 'K65N', 'D67N', 'K70R', 'L210W',
                         'T215Y', 'T215F', 'K219Q', 'K219E'],
        'didanosine':   ['K65R', 'K65E', 'K65N', 'L74V'],
        'abacavir':     ['K65R', 'K65E', 'K65N', 'L74V', 'Y115F', 'M184V'],
        'emtricitabine/lamivudine': ['K65R', 'K65E', 'K65N', 'M184V', 'M184I'],
        'tenofovir':    ['K65R', 'K65E', 'K65N', 'K70E'],
        'TAMs':         ['M41L', 'K70R', 'L210W', 'T215Y', 'T215F', 'K219Q', 'K219E'],
        '69 insertion complex': ['M41L', 'A62V', 'K70R', 'L210W', 'T215Y', 'T215F',
                                 'K219Q', 'K219E'],          # the 69 insertion itself is not a substitution
        '151 complex':  ['A62V', 'V75I', 'F77L', 'F116Y', 'Q151M'],
        'doravirine':   ['A98G', 'V106A', 'V106I', 'V106M', 'V106T', 'Y188L', 'G190E', 'P225H',
                         'F227C', 'F227I', 'F227L', 'F227R', 'F227V', 'M230L', 'L234I', 'Y318F'],
        'efavirenz':    ['L100I', 'K101P', 'K103N', 'K103S', 'V106M', 'V108I', 'Y181C', 'Y181I',
                         'Y188L', 'G190S', 'G190A', 'P225H', 'M230L'],
        'etravirine':   ['V90I', 'A98G', 'L100I', 'K101E', 'K101H', 'K101P', 'V106I',
                         'E138A', 'E138G', 'E138K', 'E138Q', 'V179D', 'V179F', 'V179T',
                         'Y181C', 'Y181I', 'Y181V', 'G190S', 'G190A', 'M230L'],
        'nevirapine':   ['L100I', 'K101P', 'K103N', 'K103S', 'V106A', 'V106M', 'V108I',
                         'Y181C', 'Y181I', 'Y188C', 'Y188L', 'Y188H', 'G190A', 'M230L'],
        'rilpivirine':  ['L100I', 'K101E', 'K101P', 'E138A', 'E138G', 'E138K', 'E138Q', 'E138R',
                         'V179L', 'Y181C', 'Y181I', 'Y181V', 'Y188L', 'H221Y', 'F227C',
                         'M230I', 'M230L'],
    },
}

print({P: sum(len(v) for v in SOURCE_B[P].values()) for P in PROTEINS},
      "drug-substitution entries")
print("drugs present in both sources:",
      sum(len(set(SOURCE_A[P]) & set(SOURCE_B[P])) for P in PROTEINS))

{'IN': 85, 'PR': 172, 'RT': 138} drug-substitution entries
drugs present in both sources: 27


## Build the pair tables

One function, run on both sources. A pair survives when both substitutions reduce cleanly, they sit
at different positions, and **at least one sequence in the MSA carries both reduced mutant
residues**. Pairs are de-duplicated after reduction, since several substitutions can collapse onto
one reduced letter (Q148H/K/R, the T215 family).

In [5]:
def build(source, label):
    """All within-drug DRM pairs of a source, with their range, mean dE double and consensus ddE."""
    rows, audit, cache = [], [], {}
    for P in PROTEINS:
        cfg = CFG[P]
        for drug, subs in source[P].items():
            good, dropped = [], []
            for s in subs:
                red = reduced_mutation(P, s)
                (good if red else dropped).append((s, red) if red else s)
            uniq = {}
            for (na, ra), (nb, rb) in itertools.combinations(good, 2):
                if ra[0] == rb[0]:
                    continue
                uniq.setdefault(tuple(sorted([ra, rb])), []).append(f"{na}-{nb}")
            kept = 0
            for (a, b), names in uniq.items():
                count = int(((cfg['codes'][:, a[0]] == a[2])
                             & (cfg['codes'][:, b[0]] == b[2])).sum())
                if count == 0:
                    continue
                key = (P, a, b)
                if key not in cache:
                    cache[key] = metrics(P, a, b)
                rng, mean, dde = cache[key]
                kept += 1
                rows.append({'p': P, 'd': drug, 'm': names[0],
                             'rp': f"{cfg['min_pos']+a[0]}-{cfg['min_pos']+b[0]}",
                             'n': count, 'r': round(rng, 3), 'e': round(dde, 3),
                             'mean': round(mean, 3), 'a': len(names)})
            audit.append({'source': label, 'protein': P, 'drug': drug,
                          'substitutions': len(subs), 'usable': len(good),
                          'lost in reduction': len(dropped),
                          'candidate pairs': len(uniq), 'kept (count > 0)': kept,
                          'dropped': ', '.join(dropped)})
    return pd.DataFrame(rows), pd.DataFrame(audit)


TAB_A, AUD_A = build(SOURCE_A, 'stanford x IAS positions')
TAB_B, AUD_B = build(SOURCE_B, 'IAS-USA only')

for nm, t in [('Stanford x IAS positions', TAB_A), ('IAS-USA only', TAB_B)]:
    print(f"{nm:<26} {len(t):>5} drug-pair rows   "
          f"{t.groupby(['p','rp']).ngroups:>4} distinct reduced position pairs   "
          f"{t['d'].nunique():>2} drugs   ddE {t['e'].min():.2f} to {t['e'].max():.2f}")

pd.set_option('display.max_rows', 60, 'display.width', 200, 'display.max_colwidth', 46)
display(pd.concat([AUD_A, AUD_B]).reset_index(drop=True))

Stanford x IAS positions    2227 drug-pair rows    338 distinct reduced position pairs   27 drugs   ddE -3.68 to 8.51
IAS-USA only                1834 drug-pair rows    426 distinct reduced position pairs   27 drugs   ddE -3.68 to 8.51


,source,protein,drug,substitutions,usable,lost in reduction,candidate pairs,kept (count > 0),dropped
0,stanford x IAS positions,IN,bictegravir,15,13,2,38,16,"G118R, S153F"
1,stanford x IAS positions,IN,cabotegravir,24,19,5,110,35,"L74F, G118R, S153F, N155S, N155T"
2,stanford x IAS positions,IN,dolutegravir,19,15,4,59,23,"G118R, S153F, N155S, N155T"
3,stanford x IAS positions,IN,elvitegravir,26,22,4,142,35,"G118R, F121C, N155S, N155T"
4,stanford x IAS positions,IN,raltegravir,31,26,5,142,54,"L74F, G118R, F121C, N155S, N155T"
5,stanford x IAS positions,PR,atazanavir,47,44,3,286,251,"I50L, I84A, I84C"
6,stanford x IAS positions,PR,darunavir,19,16,3,52,49,"I50L, I84A, I84C"
7,stanford x IAS positions,PR,fosamprenavir,33,30,3,143,138,"I50L, I84A, I84C"
8,stanford x IAS positions,PR,indinavir,33,31,2,160,154,"I84A, I84C"
9,stanford x IAS positions,PR,lopinavir,39,36,3,242,233,"I50L, I84A, I84C"


## Write the pages

One template, filled twice. The page is self-contained — data embedded, no CDN — so it opens from
disk.

In [6]:
TEMPLATE = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<style>
  :root{
    --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
    --grid:#e6e5df; --base:#c3c2b7;
    --IN:#175d92; --PR:#6d0f18; --RT:#c98f2b;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--surface);color:var(--ink);
       font:13px/1.45 -apple-system,BlinkMacSystemFont,"Segoe UI",Helvetica,Arial,sans-serif;}
  .wrap{max-width:1180px;margin:0 auto;padding:22px 26px 60px}
  h1{font-size:17px;font-weight:600;margin:0 0 4px}
  .sub{color:var(--ink2);font-size:12px;margin:0 0 18px;max-width:80ch}
  .panel{border:1px solid var(--grid);border-radius:8px;padding:14px 16px;margin-bottom:18px;
         display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:14px 26px;
         align-items:start;background:#fff}
  .ctl label{display:block;font-size:11px;letter-spacing:.04em;text-transform:uppercase;
             color:var(--muted);margin-bottom:6px}
  .ctl .val{font-variant-numeric:tabular-nums;color:var(--ink);font-weight:600}
  input[type=range]{width:100%;accent-color:#6d0f18;margin:2px 0}
  .row-ctl{display:flex;gap:16px;align-items:center;flex-wrap:wrap;font-size:12px;color:var(--ink2)}
  button{font:inherit;padding:5px 11px;border:1px solid var(--base);background:#fff;
         border-radius:5px;cursor:pointer;color:var(--ink2)}
  button:hover{border-color:var(--ink2);color:var(--ink)}
  .legend{display:flex;gap:14px;align-items:center;font-size:12px;color:var(--ink2)}
  .sw{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:5px;
      vertical-align:-1px}
  .stat{font-variant-numeric:tabular-nums;color:var(--ink2);font-size:12px;margin:0 0 10px}
  .stat b{color:var(--ink)}
  svg{display:block;width:100%;height:auto;overflow:visible}
  .rowg{transition:transform .45s cubic-bezier(.3,.7,.3,1)}
  .dot{transition:opacity .25s linear;cursor:crosshair}
  .dot.off{opacity:0 !important;pointer-events:none}
  .boxr,.med,.whisk,.cap{transition:opacity .25s linear}
  .lab{font-size:11px;fill:var(--ink2)}
  .nlab{font-size:10px;fill:var(--muted);font-variant-numeric:tabular-nums}
  .ax{font-size:10px;fill:var(--muted)}
  .empty .lab,.empty .nlab{fill:#cfcec7}
  #tip{position:fixed;pointer-events:none;opacity:0;transition:opacity .12s;z-index:9;
       background:#111;color:#fff;padding:7px 9px;border-radius:5px;font-size:11.5px;
       line-height:1.5;white-space:nowrap;box-shadow:0 3px 14px rgba(0,0,0,.25)}
  #tip b{font-weight:600}
  .note{color:var(--muted);font-size:11.5px;max-width:82ch;margin-top:22px}
</style>
</head>
<body>
<div class="wrap">
  <h1>__H1__</h1>
  <p class="sub">__SUB__</p>

  <div class="panel">
    <div class="ctl">
      <label>&Delta;&Delta;E on consensus &ge; <span class="val" id="loLab"></span></label>
      <input type="range" id="lo" step="0.01">
      <div class="row-ctl" style="margin-top:6px">
        <span id="eRange"></span>
      </div>
    </div>
    <div class="ctl">
      <label>bivariate count in MSA &ge; <span class="val" id="cntLab"></span></label>
      <input type="range" id="cnt" min="1" max="200" step="1" value="1">
      <label style="margin-top:8px">show</label>
      <div class="row-ctl">
        <label style="text-transform:none;letter-spacing:0;font-size:12px;color:var(--ink2)">
          <input type="checkbox" id="sort" checked> re-sort rows by median</label>
        <label style="text-transform:none;letter-spacing:0;font-size:12px;color:var(--ink2)">
          <input type="checkbox" id="hide"> hide empty drugs</label>
      </div>
    </div>
    <div class="ctl">
      <label>protein</label>
      <div class="row-ctl" id="protBox"></div>
      <div style="margin-top:12px"><button id="reset">reset all</button></div>
    </div>
  </div>

  <p class="stat" id="stat"></p>
  <svg id="chart"></svg>
  <p class="note">Box = IQR across the drug's surviving pairs, heavy line = median, whiskers = the
  furthest pair within 1.5&times;IQR, dots = the pairs themselves (jittered vertically only; the x
  value is exact). &Delta;&Delta;E here is evaluated on the consensus sequence, the same convention
  as the ms0_5 ddE tables &mdash; G140S-Q148H returns 8.51. A pair reachable from several drugs is
  drawn once per drug with the same x. __SRC__</p>
</div>
<div id="tip"></div>

<script id="data" type="application/json">__PAYLOAD__</script>
<script>
const RAW = JSON.parse(document.getElementById('data').textContent);
const COMPLEX = new Set(['TAMs','69 insertion complex','151 complex']);
const COL = {IN:'#175d92', PR:'#6d0f18', RT:'#c98f2b'};
const PROTS = ['IN','PR','RT'];
const active = new Set(PROTS);

// ---- data prep: one group per (protein, drug), dots carry a fixed jitter ----------------
let seed = 7;
const rnd = () => (seed = (seed*1103515245 + 12345) & 0x7fffffff) / 0x7fffffff;
const groups = [];
const byKey = new Map();
for (const r of RAW) {
  const key = r.p + '|' + r.d;
  let g = byKey.get(key);
  if (!g) { g = {prot:r.p, drug:r.d, dots:[]}; byKey.set(key, g); groups.push(g); }
  g.dots.push({...r, j:(rnd()-0.5)*0.62});
}
// baseline order: median range over all pairs, widest first -- what the row order falls back
// to when live re-sorting is switched off
{
  const med = a => { const s = a.slice().sort((x,y)=>x-y), i = (s.length-1)/2;
    return s.length % 2 ? s[i] : (s[i-0.5] + s[i+0.5]) / 2; };
  groups.sort((a,b) => med(b.dots.map(d=>d.r)) - med(a.dots.map(d=>d.r)));
  groups.forEach((g,i) => g.base = i);
}

const eAll = RAW.map(r => r.e);
const E_MIN = Math.floor(Math.min(...eAll)*10)/10, E_MAX = Math.ceil(Math.max(...eAll)*10)/10;
const R_MIN = 0, R_MAX = Math.ceil(Math.max(...RAW.map(r => r.r)));

// ---- layout -----------------------------------------------------------------------------
const M = {t:26, r:58, b:44, l:190}, RH = 30;
const W = 1130, PW = W - M.l - M.r;
const svg = document.getElementById('chart');
const H = M.t + M.b + groups.length*RH;
svg.setAttribute('viewBox', `0 0 ${W} ${H}`);
const NS = 'http://www.w3.org/2000/svg';
const el = (n, a) => { const e = document.createElementNS(NS, n);
  for (const k in a) e.setAttribute(k, a[k]); return e; };
const X = v => M.l + (v - R_MIN) / (R_MAX - R_MIN) * PW;

// grid + axis
const gAx = el('g', {});
for (let v = R_MIN; v <= R_MAX; v += 2.5) {
  gAx.appendChild(el('line', {x1:X(v), x2:X(v), y1:M.t-6, y2:H-M.b+4,
                              stroke:'#e6e5df', 'stroke-width':1}));
  const t = el('text', {x:X(v), y:H-M.b+18, 'text-anchor':'middle', class:'ax'});
  t.textContent = v.toFixed(1); gAx.appendChild(t);
}
const xt = el('text', {x:M.l+PW/2, y:H-M.b+34, 'text-anchor':'middle', class:'ax'});
xt.textContent = 'dE double range (95-5%) of the pair, over the whole alignment';
gAx.appendChild(xt);
svg.appendChild(gAx);

// one <g> per drug row
for (const g of groups) {
  g.g = el('g', {class:'rowg'});
  g.whisk = el('line', {class:'whisk', stroke:'#c3c2b7', 'stroke-width':1});
  g.cap1  = el('line', {class:'cap', stroke:'#c3c2b7', 'stroke-width':1});
  g.cap2  = el('line', {class:'cap', stroke:'#c3c2b7', 'stroke-width':1});
  g.box   = el('rect', {class:'boxr', fill:COL[g.prot], 'fill-opacity':.2,
                        stroke:COL[g.prot], 'stroke-width':1.1, rx:1.5});
  g.med   = el('line', {class:'med', stroke:'#0b0b0b', 'stroke-width':1.8});
  g.lab   = el('text', {x:M.l-12, y:4, 'text-anchor':'end', class:'lab'});
  g.lab.textContent = COMPLEX.has(g.drug) ? g.drug + ' (complex)' : g.drug;
  g.nlab  = el('text', {x:M.l+PW+8, y:4, class:'nlab'});
  g.g.append(g.whisk, g.cap1, g.cap2, g.box, g.med, g.lab, g.nlab);
  for (const d of g.dots) {
    d.c = el('circle', {class:'dot', cx:X(d.r), cy:d.j*RH*0.5, r:2.6,
                        fill:COL[g.prot], 'fill-opacity':.5});
    d.c.addEventListener('mouseenter', ev => showTip(ev, g, d));
    d.c.addEventListener('mouseleave', hideTip);
    g.g.appendChild(d.c);
  }
  svg.appendChild(g.g);
}

// ---- stats -------------------------------------------------------------------------------
const quant = (s, q) => { const i = (s.length-1)*q, lo = Math.floor(i), hi = Math.ceil(i);
  return lo === hi ? s[lo] : s[lo] + (s[hi]-s[lo])*(i-lo); };

function render() {
  const lo = +eLo.value, minN = +cnt.value;
  let shown = 0;
  for (const g of groups) {
    const live = [];
    for (const d of g.dots) {
      const on = active.has(g.prot) && d.e >= lo && d.n >= minN;
      d.c.classList.toggle('off', !on);
      if (on) live.push(d.r);
    }
    live.sort((a,b) => a-b);
    g.n = live.length; shown += live.length;
    if (live.length) {
      g.q1 = quant(live,.25); g.q2 = quant(live,.5); g.q3 = quant(live,.75);
      const iqr = g.q3-g.q1;
      g.w1 = live.find(v => v >= g.q1 - 1.5*iqr);
      g.w2 = [...live].reverse().find(v => v <= g.q3 + 1.5*iqr);
    } else { g.q2 = -1; }
  }
  const order = groups.slice();
  // y grows downward, so descending median puts the widest drug at the top, matching
  // the static figure. Drugs with nothing left sink to the bottom rather than the top.
  const key = g => g.n ? g.q2 : -Infinity;
  order.sort((a,b) => sortCb.checked ? key(b) - key(a) : a.base - b.base);
  let k = 0;
  for (const g of order) {
    const empty = g.n === 0;
    const vis = !(empty && hideCb.checked);
    g.g.style.display = vis ? '' : 'none';
    if (vis) { g.g.setAttribute('transform', `translate(0,${M.t + (k+0.5)*RH})`); k++; }
    g.g.classList.toggle('empty', empty);
    for (const n of [g.box, g.med, g.whisk, g.cap1, g.cap2])
      n.setAttribute('opacity', empty ? 0 : 1);
    g.nlab.textContent = empty ? '--' : 'n=' + g.n;
    if (empty) continue;
    const h = 15;
    g.box.setAttribute('x', X(g.q1)); g.box.setAttribute('width', Math.max(X(g.q3)-X(g.q1), 1));
    g.box.setAttribute('y', -h/2);    g.box.setAttribute('height', h);
    g.med.setAttribute('x1', X(g.q2)); g.med.setAttribute('x2', X(g.q2));
    g.med.setAttribute('y1', -h/2);    g.med.setAttribute('y2', h/2);
    g.whisk.setAttribute('x1', X(g.w1)); g.whisk.setAttribute('x2', X(g.w2));
    g.whisk.setAttribute('y1', 0); g.whisk.setAttribute('y2', 0);
    for (const [c, v] of [[g.cap1, g.w1], [g.cap2, g.w2]]) {
      c.setAttribute('x1', X(v)); c.setAttribute('x2', X(v));
      c.setAttribute('y1', -5); c.setAttribute('y2', 5);
    }
  }
  const drugs = groups.filter(g => g.n > 0).length;
  stat.innerHTML = `<b>${shown}</b> of ${RAW.length} drug-pair points shown, across ` +
    `<b>${drugs}</b> of ${groups.length} drugs &nbsp;&middot;&nbsp; ` +
    `ddE &ge; ${lo.toFixed(2)}, bivariate count &ge; ${minN}`;
}

// ---- tooltip -----------------------------------------------------------------------------
function showTip(ev, g, d) {
  const tip = document.getElementById('tip');
  tip.innerHTML = `<b>${d.m}</b> &nbsp;<span style="opacity:.65">${g.prot} ${d.rp}</span><br>` +
    `dE double range &nbsp;<b>${d.r.toFixed(2)}</b><br>` +
    `ddE on consensus &nbsp;<b>${d.e.toFixed(2)}</b><br>` +
    `both mutations in &nbsp;<b>${d.n}</b> sequences` +
    (d.a > 1 ? `<br><span style="opacity:.65">${d.a} Stanford pairs share this reduced pair</span>` : '');
  tip.style.opacity = 1;
  tip.style.left = Math.min(ev.clientX + 14, innerWidth - 240) + 'px';
  tip.style.top = (ev.clientY + 14) + 'px';
}
function hideTip() { document.getElementById('tip').style.opacity = 0; }   // declaration, not const: the row loop
                                                 // above wires it before this line runs

// ---- controls ----------------------------------------------------------------------------
const eLo = document.getElementById('lo'),
      cnt = document.getElementById('cnt'), sortCb = document.getElementById('sort'),
      hideCb = document.getElementById('hide'), stat = document.getElementById('stat');
eLo.min = E_MIN; eLo.max = E_MAX; eLo.value = E_MIN;
document.getElementById('eRange').textContent =
  `ddE runs ${E_MIN.toFixed(2)} to ${E_MAX.toFixed(2)} over the ${RAW.length} points`;
const labels = () => {
  document.getElementById('loLab').textContent = (+eLo.value).toFixed(2);
  document.getElementById('cntLab').textContent = cnt.value;
};
const upd = () => { labels(); render(); };
for (const s of [eLo, cnt]) s.addEventListener('input', upd);
for (const c of [sortCb, hideCb]) c.addEventListener('change', render);

const pb = document.getElementById('protBox');
for (const p of PROTS) {
  const l = document.createElement('label');
  l.style.cssText = 'text-transform:none;letter-spacing:0;font-size:12px';
  l.innerHTML = `<input type="checkbox" checked data-p="${p}">` +
                `<span class="sw" style="background:${COL[p]}"></span>${p}`;
  l.querySelector('input').addEventListener('change', e => {
    e.target.checked ? active.add(p) : active.delete(p); render();
  });
  pb.appendChild(l);
}
document.getElementById('reset').addEventListener('click', () => {
  eLo.value = E_MIN; cnt.value = 1;
  sortCb.checked = true; hideCb.checked = false;
  for (const i of pb.querySelectorAll('input')) i.checked = true;
  active.clear(); PROTS.forEach(p => active.add(p));
  upd();
});
upd();
</script>
</body>
</html>
"""

In [7]:
def write_page(table, path, h1, sub, src_note, title):
    """Fill the template with one source's rows and write a standalone HTML page."""
    cols = ['p', 'd', 'm', 'rp', 'n', 'r', 'e', 'a']
    payload = json.dumps(table[cols].to_dict('records'), separators=(',', ':'))
    html = (TEMPLATE.replace('__TITLE__', title).replace('__H1__', h1)
            .replace('__SUB__', sub).replace('__SRC__', src_note)
            .replace('__PAYLOAD__', payload))
    Path(path).write_text(html)
    print(f"wrote {path}  {len(html)/1024:.0f} KB  {len(table)} points")


SUB_COMMON = ("The x position is the 95&ndash;5 percentile spread of &Delta;E double over the whole "
              "alignment. Raise the &Delta;&Delta;E threshold and the dots drop out, the boxes "
              "recompute, and the rows re-sort on the new medians.")

write_page(
    TAB_A, out_dir / 'de_range_by_drug_interactive.html',
    "&Delta;E double range by drug &mdash; Stanford substitutions at IAS-USA positions",
    "One dot per DRM pair of that drug &mdash; every Stanford HIVDB substitution at the drug's "
    "IAS-USA codon positions, paired up, kept when the bivariate marginal in the MSA is "
    "non-zero. " + SUB_COMMON,
    "Mutations from <code>ms0_5/*/data/drug_resistant_pair_*_stanford.tsv</code>; "
    "drug assignment by IAS-USA 2025 positions.",
    "dE double range by drug - Stanford x IAS positions")

write_page(
    TAB_B, out_dir / 'de_range_by_drug_interactive_IAS.html',
    "&Delta;E double range by drug &mdash; IAS-USA 2025 only",
    "One dot per DRM pair of that drug, with both the substitutions and the drug assignment taken "
    "from the IAS-USA 2025 mutation figures, paired up, kept when the bivariate marginal in the "
    "MSA is non-zero. " + SUB_COMMON,
    "Mutations and drug assignment both from IAS-USA 2025, <i>Topics in Antiviral Medicine</i> "
    "33(2):457-465, pages 462-463.",
    "dE double range by drug - IAS-USA 2025")

TAB_A.to_csv(out_dir / 'de_range_pairs_stanford_x_ias.csv', index=False)
TAB_B.to_csv(out_dir / 'de_range_pairs_ias_only.csv', index=False)
pd.concat([AUD_A, AUD_B]).to_csv(out_dir / 'de_range_pairs_source_audit.csv', index=False)
print('wrote the two CSVs and the audit')

wrote /Users/xuechenkan/potts_model_test/ms1/observed_vs_expected/de_range_by_drug_interactive.html  215 KB  2227 points
wrote /Users/xuechenkan/potts_model_test/ms1/observed_vs_expected/de_range_by_drug_interactive_IAS.html  178 KB  1834 points
wrote the two CSVs and the audit


## How much the source choice changes the answer

Per-drug median range under each source, and the drugs where they disagree most.

In [8]:
MA = TAB_A.groupby(['p', 'd'])['r'].agg(['median', 'size']).rename(
    columns={'median': 'stanford median', 'size': 'stanford pairs'})
MB = TAB_B.groupby(['p', 'd'])['r'].agg(['median', 'size']).rename(
    columns={'median': 'IAS median', 'size': 'IAS pairs'})
CMP = MA.join(MB, how='outer').reset_index().rename(columns={'p': 'protein', 'd': 'drug'})
CMP['shift'] = CMP['IAS median'] - CMP['stanford median']

both = CMP.dropna(subset=['stanford median', 'IAS median'])
rk = lambda s: pd.Series(s).rank()
print(f"per-drug median range, Stanford vs IAS-USA construction: "
      f"Spearman rho = {np.corrcoef(rk(both['stanford median']), rk(both['IAS median']))[0,1]:.3f} "
      f"(n = {len(both)} drugs)")
print(f"median |shift| = {both['shift'].abs().median():.2f} energy units")
display(CMP.sort_values('shift').round(2))

per-drug median range, Stanford vs IAS-USA construction: Spearman rho = 0.943 (n = 27 drugs)
median |shift| = 0.26 energy units


,protein,drug,stanford median,stanford pairs,IAS median,IAS pairs,shift
25,RT,tenofovir,8.50,6,7.04,1,-1.46
11,PR,saquinavir,12.66,80,11.44,114,-1.22
12,PR,tipranavir,11.20,123,10.01,176,-1.18
6,PR,darunavir,10.12,49,9.27,43,-0.84
8,PR,indinavir,11.81,154,11.48,202,-0.33
22,RT,nevirapine,5.71,153,5.56,68,-0.15
13,RT,151 complex,5.64,23,5.53,10,-0.10
7,PR,fosamprenavir,11.58,138,11.50,112,-0.08
18,RT,doravirine,5.91,40,5.88,15,-0.03
20,RT,emtricitabine/lamivudine,4.54,6,4.54,6,0.00
